# S05 · Symbol Table Studio: Correctness + Fair Benchmarking
**M12 · Data Structures and Algorithms | MSc Computer Science | Python 3 | 3-hour guided lab**

In this studio you will explore three ways to store **unique key → value** pairs: **array**, **linked list**, and **binary search tree (BST)**. They must behave the same from the user's point of view.

> **Predict → Run → Modify → Explain.** You will see runnable demonstrations, small incomplete student tasks, contract tests, and a reproducible benchmark. This is **practice supporting Assignment 1**, not a graded assignment brief or an answer key. Official marking rules and deadlines are still pending institutional confirmation.

**All baseline cells run in Google Colab without installation.** TODO cells are safe to run until you decide to implement them.

## Learning targets and suggested classroom sequence
**By the end:** define a common `put/get/delete/contains/len` interface, explain invariants, design boundary tests, build one piece of an implementation, control benchmarking experiments, and make an evidence-based trade-off statement.

**00–10 min** orient · **10–35** ADT and representations · **35–70** test and benchmarking demo · **70–165** studio work · **165–180** review and exit. The lecturer may adjust the pace and locally required breaks.

### First prediction
If you call `put(20, 'A')` and then `put(20, 'C')`, how many bindings should be in the table? Write a prediction before running anything.

## 1. What is a symbol table?
A **symbol table** is an Abstract Data Type (ADT): it describes **what happens**, not how the information is stored.

Teaching-only contract for our practice:

| Operation | Expected behaviour |
|---|---|
| `put(key, value)` | Add a new key, or replace the value for an existing key |
| `get(key)` | Return associated value; raise `KeyError` if missing |
| `delete(key)` | Remove binding; raise `KeyError` if missing |
| `contains(key)` | Return `True` only if key exists |
| `len(table)` | Number of *unique* keys |

We use comparable **integer** keys. Values may be `None`. The official graded API may differ: this is an illustrative consistent API.

In [ ]:
# Working reference for observable behaviour. Python dict is NOT an Assignment 1 implementation.
class DictReference:
    def __init__(self):
        self.data = {}
    def put(self, key, value):
        self.data[key] = value
    def get(self, key):
        return self.data[key]
    def delete(self, key):
        del self.data[key]
    def contains(self, key):
        return key in self.data
    def __len__(self):
        return len(self.data)

reference = DictReference()
reference.put(20, 'A')
reference.put(40, 'B')
reference.put(20, 'C')
print('get(20):', reference.get(20))
print('size:', len(reference))
print('contains(99):', reference.contains(99))
assert reference.get(20) == 'C' and len(reference) == 2

**Explain:** Why did size stay at 2? What would go wrong if the linked-list representation appended a new node for key `20` on every update?

In [ ]:
# Step-by-step trace of an absent-key contract
try:
    reference.get(99)
except KeyError:
    print('Missing key correctly raises KeyError.')
reference.put(7, None)
print('Key 7 exists:', reference.contains(7), '| value:', reference.get(7))
assert reference.contains(7) and reference.get(7) is None

## 2. Same data, three representations
Imagine keys `40, 20, 60, 10` were inserted in that order with four associated values.

**Unsorted array of pairs:** `[(40,'A'), (20,'B'), (60,'C'), (10,'D')]`. A lookup scans entries; the invariant is **no duplicate keys**.

**Linked list:** `HEAD → [10:D] → [60:C] → [20:B] → [40:A] → None` if new keys are inserted at the head. A lookup follows links; invariant: the chain is finite, acyclic, and contains **no duplicate keys**.

**Plain BST:** each node splits smaller and larger keys. Invariant: **every key** left of a node is smaller, **every key** right of it is larger, recursively. A plain BST need not be balanced.

The external `get(20)` must return the same value regardless of storage.

In [ ]:
# A complete, tiny teaching demonstration of LOOKUP ONLY in a list of pairs.
# This is not a finished array symbol-table implementation.
pairs = [(40, 'A'), (20, 'B'), (60, 'C'), (10, 'D')]

def demonstrate_linear_lookup(entries, target):
    comparisons = 0
    for key, value in entries:
        comparisons += 1
        if key == target:
            return value, comparisons
    raise KeyError(target)

for target in [40, 10]:
    print('lookup', target, '→', demonstrate_linear_lookup(pairs, target))

**Predict before modifying:** Which key above needs the most comparisons? Why does array storage not automatically make key lookup O(1)? Try a different order of pairs and write down what changed.

In [ ]:
# Linked-list traversal illustration, not a complete linked symbol table.
from dataclasses import dataclass
from typing import Optional

@dataclass
class TeachingListNode:
    key: int
    value: str
    next: Optional['TeachingListNode'] = None

head = TeachingListNode(10, 'D', TeachingListNode(60, 'C', TeachingListNode(20, 'B', TeachingListNode(40, 'A'))))
current = head
visited = []
while current is not None:
    visited.append(current.key)
    current = current.next
print('Keys reached by next pointers:', visited)
assert visited == [10, 60, 20, 40]

**Modification task:** Draw which `next` pointer must change to delete key 60. Think about deleting the *head* separately. What error happens if the deletion accidentally loses the rest of the chain?

In [ ]:
# BST search trace only; no full put/delete solution is provided.
@dataclass
class TeachingBSTNode:
    key: int
    value: str
    left: Optional['TeachingBSTNode'] = None
    right: Optional['TeachingBSTNode'] = None

root = TeachingBSTNode(40, 'A',
    left=TeachingBSTNode(20, 'B', left=TeachingBSTNode(10, 'D')),
    right=TeachingBSTNode(60, 'C'))

def demonstrate_bst_search_path(root, target):
    node, path = root, []
    while node is not None:
        path.append(node.key)
        if target == node.key:
            return node.value, path
        node = node.left if target < node.key else node.right
    raise KeyError((target, path))

for target in [10, 60]:
    print('BST get', target, '→', demonstrate_bst_search_path(root, target))

**Explain:** Why can a BST skip one entire subtree after a comparison? Why does its lookup take O(h+1) instead of always O(log n), with h measured in edges? What happens if the keys arrive sorted?

## 3. Write tests before implementing
A **test** specifies an input and expected observable result. A passing test is useful evidence, not proof for all inputs.

Start with: empty lookup, duplicate-key update, stored `None` value, missing deletion, head/root deletion, and BST leaf/one/two-child deletion.

**Important:** all three implementations must use the same missing-key and size rules or measured behaviour is not comparable.

In [ ]:
# A reusable PUBLIC contract self-check; it runs against the built-in reference now.
def basic_contract_check(factory):
    t = factory()
    assert len(t) == 0
    assert not t.contains(8)
    try:
        t.get(8)
    except KeyError:
        pass
    else:
        raise AssertionError('Missing get did not raise KeyError')
    t.put(8, 'old')
    t.put(3, 'blue')
    t.put(8, None)
    assert len(t) == 2 and t.contains(8)
    assert t.get(8) is None and t.get(3) == 'blue'
    t.delete(3)
    assert len(t) == 1 and not t.contains(3)
    try:
        t.delete(3)
    except KeyError:
        pass
    else:
        raise AssertionError('Missing delete did not raise KeyError')
    return 'pass'

print('Reference contract:', basic_contract_check(DictReference))

### Your test ideas (fill in)
- For an **array**, show an operation sequence that detects duplicate bindings.
- For a **linked list**, show a delete operation that checks the tail was not lost.
- For a **BST**, use keys `[50,30,70,20,40,60,80]` and delete 50. Check all remaining **key–value** pairs.
- For all structures, why is `contains(key)` different from `get(key) is not None`?

## 4. Your implementation studio — deliberately incomplete
The next cell defines class signatures to guide your own implementation. They intentionally raise `NotImplementedError`. **No full assessed solution is given.** When you have implemented a method, replace its TODO. Start with `get`; implement and test one operation at a time.

This cell only *defines* the starter classes and therefore runs without calling unfinished operations.

In [ ]:
class StudentArrayTable:
    def __init__(self):
        self.entries = []
    def put(self, key, value):
        # TODO: search existing key; update or add without duplicates
        raise NotImplementedError('Implement array put')
    def get(self, key):
        # TODO: scan entries; raise KeyError on a missing key
        raise NotImplementedError('Implement array get')
    def delete(self, key):
        # TODO: find and remove matching binding
        raise NotImplementedError('Implement array delete')
    def contains(self, key):
        raise NotImplementedError('Implement array contains')
    def __len__(self):
        raise NotImplementedError('Implement array size')

class StudentLinkedTable:
    def __init__(self):
        self.head = None
        self.size = 0
    def put(self, key, value):
        raise NotImplementedError('Implement linked put')
    def get(self, key):
        raise NotImplementedError('Implement linked get')
    def delete(self, key):
        raise NotImplementedError('Implement linked delete')
    def contains(self, key):
        raise NotImplementedError('Implement linked contains')
    def __len__(self):
        return self.size

class StudentBSTTable:
    def __init__(self):
        self.root = None
        self.size = 0
    def put(self, key, value):
        raise NotImplementedError('Implement BST put')
    def get(self, key):
        raise NotImplementedError('Implement BST get')
    def delete(self, key):
        raise NotImplementedError('Implement BST delete')
    def contains(self, key):
        raise NotImplementedError('Implement BST contains')
    def __len__(self):
        return self.size

print('Student class scaffolds are ready; methods are intentionally incomplete.')

**Step-by-step student task**
1. Choose the **array** representation. In words, define how to find an existing key without duplicating it.
2. Implement `get`, then run your own two-key lookup and missing-key checks.
3. Implement `put`: define what happens when a key already exists before inserting anything new.
4. Implement `delete` and `__len__`; test empty, singleton and missing keys.
5. Repeat for the **linked list**: draw `head`, `current`, `previous`; do not forget head deletion.
6. Repeat for the **BST**: preserve the global ordering rule; test successor replacement for a two-child deletion.
7. Use the public contract check only **after** your implementation is complete.

**No need to finish all three during one 95-minute block; document progress accurately.**

In [ ]:
# Safe gate: change the next flag to True only AFTER completing StudentArrayTable.
READY_ARRAY = False
if READY_ARRAY:
    print('Array implementation:', basic_contract_check(StudentArrayTable))
else:
    print('Array tests deferred until TODOs are complete — expected starter behaviour.')

## 5. Why insertion order changes a plain BST
BST height is the **maximum number of edges from root to a leaf**. A plain BST built with sorted keys can become a chain. We can measure the shape **without writing a complete BST map** by counting the comparisons needed to find the insertion position in a simplified *tree of keys*.

This small demonstration only counts insertion path length, not a finished map implementation.

In [ ]:
import random

def demo_shape_statistics(keys):
    # Small teaching-only BST shape simulator: dictionary of nodes with child links.
    if not keys:
        return 0, 0
    nodes = {keys[0]: {'left': None, 'right': None}}
    root_key = keys[0]
    max_depth = 0
    comparisons = 0
    for key in keys[1:]:
        node_key, depth = root_key, 0
        while True:
            comparisons += 1
            direction = 'left' if key < node_key else 'right'
            child = nodes[node_key][direction]
            if child is None:
                nodes[node_key][direction] = key
                nodes[key] = {'left': None, 'right': None}
                max_depth = max(max_depth, depth + 1)
                break
            node_key, depth = child, depth + 1
    return max_depth, comparisons

n = 16
sorted_keys = list(range(n))
shuffled_keys = sorted_keys.copy()
random.Random(42).shuffle(shuffled_keys)
print('sorted:', demo_shape_statistics(sorted_keys))
print('shuffled seed=42:', demo_shape_statistics(shuffled_keys))

**Predict, modify, explain:** Set `n=8`, then `n=32`. What does sorted insertion do to height? Is the exact shuffled result guaranteed for every random seed? Explain why we can analyse worst case without asserting that random order guarantees a perfectly balanced tree.

## 6. Fair benchmarks: compare the same job
**Fairness rules**
1. Check correctness first.
2. Choose the same n, keys, workload and query sequence for every implementation.
3. Keep **setup outside the timed lookup region**.
4. Repeat measurements; record median and units. A warm-up run can reduce first-call effects.
5. Declare seed, hardware/runtime, operation count and input distribution.
6. Time only completed, correct student implementations. **Never invent results**.

We will first compare two **teaching-only lookup mechanisms** (list scan and `dict`), neither of which is the complete graded three-table comparison. This explains how to isolate a workload.

In [ ]:
import platform, statistics, sys, time

def timed_batch(operation, queries):
    checksum = 0
    start = time.perf_counter_ns()
    for k in queries:
        checksum += operation(k)
    elapsed_ns = time.perf_counter_ns() - start
    return elapsed_ns, checksum

def list_scan_value(pairs, target):
    for key, value in pairs:
        if key == target:
            return value
    raise KeyError(target)

def benchmark_lookup_demo(n, repeats=5, seed=42):
    keys = list(range(n))
    random.Random(seed).shuffle(keys)
    pairs = [(k, k) for k in keys]   # setup, not timed
    lookup_dict = dict(pairs)          # setup, not timed
    queries = list(keys)              # identical successful queries
    expected = sum(queries)
    results = {}
    for name, operation in [
        ('list scan (illustration)', lambda k: list_scan_value(pairs, k)),
        ('Python dict baseline', lambda k: lookup_dict[k]),
    ]:
        timed_batch(operation, queries)  # warm-up
        samples = []
        for _ in range(repeats):
            ns, checksum = timed_batch(operation, queries)
            assert checksum == expected
            samples.append(ns / n / 1000.0)  # microseconds per operation
        results[name] = statistics.median(samples)
    return results

print('Python:', sys.version.split()[0], '| OS:', platform.system())
for n in [100, 500, 1000]:
    result = benchmark_lookup_demo(n, repeats=5, seed=42)
    print(f'n={n:4}:', {k: round(v, 3) for k, v in result.items()}, 'µs/get')

**Do not copy any particular numbers into your report as universal facts.** The values above were measured *on the runtime that executed them* and may vary between Colab sessions. Explain why lookup data setup belongs outside the timed region, and why repeated runs may differ. Note that this compares **two illustrative lookup functions**, not your three complete symbol-table implementations.

### Student extension: benchmark your three implementations
Once all operations and contract checks pass, create a factory for each implemented table. Measure **building**, **successful lookup**, **unsuccessful lookup**, **update** and **delete** as separate workloads, with the same setup and key sequences. Keep sorted and shuffled input distributions separate. For miss lookup, preselect keys absent from every table; measure the expected exception handling consistently.

**Warning:** Keep n small enough to finish in class. Testing a degenerate BST with recursive algorithms on large n may hit Python's recursion limit. This is a memory/implementation issue, not proof of asymptotic running time.

In [ ]:
# Template for recording REAL observations. Blank measurements are deliberate.
experiment_plan = [
    {'structure': 'array', 'n': 100, 'workload': 'hit-get', 'median_us_op': None, 'correctness_passed': None},
    {'structure': 'linked', 'n': 100, 'workload': 'hit-get', 'median_us_op': None, 'correctness_passed': None},
    {'structure': 'BST', 'n': 100, 'workload': 'hit-get', 'median_us_op': None, 'correctness_passed': None},
]
for row in experiment_plan:
    print(row)
print('Fill timing fields only after testing completed implementations.')

## 7. Make a defensible engineering conclusion
Example *format* (not a pre-written result):

> For **[workload]** at n=**[size range]** using **[input order]**, I observed **[measured data]**. The algorithmic model predicts **[cost with assumptions]** because **[invariant/height/search argument]**. A limitation is **[runtime noise, memory effect, limited size, input distribution, incomplete implementation]**. Therefore, I would choose **[structure]** if **[conditions]**.

A result table should distinguish real data, unsupported hypotheses, and unmeasured structures. A benchmark curve **does not prove** Big-O by itself.

## 8. Exit ticket · 3 short answers
1. **Invariant:** what condition guarantees unique keys in an array or linked list, and what global condition defines a BST?
2. **Correctness:** describe one test that catches a key/value bug during BST two-child deletion.
3. **Fairness:** why must two symbol tables receive the *same* inputs and operation sequence for comparison?

**Optional next steps:** review S04 deletion; implement one complete table at a time; measure sorted versus shuffled input; document experiment metadata. Follow only the institution-approved Assignment 1 brief once issued.

## References and responsible interpretation
- Approved `course-plan.md`, S05: Assignment 1 studio — array, linked-list and BST symbol tables; correctness and empirical comparison.
- Cormen, Leiserson, Rivest & Stein (2022), *Introduction to Algorithms*, 4th ed.
- Goodrich, Tamassia & Goldwasser (2013), *Data Structures and Algorithms in Python*.
- Sedgewick & Wayne (2011), *Algorithms*, 4th ed.

This notebook deliberately contains incomplete assignment scaffolds and **no instructor answer key**. Official marking weights within the 40% coursework component, submission dates, rubrics and graded-language rules remain pending institutional confirmation.